In [ ]:
!pip install nltk --quiet

In [ ]:
import io
import random
import string
import warnings
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('popular', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [ ]:
# Upload chatbot.txt first using this code
from google.colab import files

uploaded = files.upload()

# Read the uploaded file
with open('Chatbot.txt', 'r', encoding='utf8', errors='ignore') as f:
    raw = f.read().lower()

Saving Chatbot.txt to Chatbot.txt


In [ ]:
# Split corpus into sentences and words

sent_tokens = nltk.sent_tokenize(raw)   # list of sentences
word_tokens = nltk.word_tokenize(raw)   # list of words


In [ ]:
# ── PREPROCESSING (FIXED) ──

lemmer = WordNetLemmatizer()

def LemTokens(tokens):
    return [lemmer.lemmatize(token) for token in tokens]

remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)

def LemNormalize(text):
    """Full pipeline: lowercase → remove punctuation → tokenize → lemmatize"""
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = nltk.word_tokenize(text.lower().translate(remove_punct_dict))
    return LemTokens(tokens)

In [ ]:
GREETING_INPUTS   = ("hello", "hi", "greetings", "sup", "what's up", "hey")
GREETING_RESPONSES = ["hi", "hey", "*nods*", "hi there", "hello", "I am glad! You are talking to me"]

def greeting(sentence):
    for word in sentence.split():
        if word.lower() in GREETING_INPUTS:
            return random.choice(GREETING_RESPONSES)
    return None

In [ ]:
# ── RESPONSE FUNCTION (FIXED) ──

def response(user_response):
    robo_response = ''
    sent_tokens.append(user_response)

    try:
        TfidfVec = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english')
        tfidf    = TfidfVec.fit_transform(sent_tokens)
        vals     = cosine_similarity(tfidf[-1], tfidf)
        idx      = vals.argsort()[0][-2]
        flat     = vals.flatten()
        flat.sort()
        req_tfidf = flat[-2]

        if req_tfidf == 0:
            robo_response = "I am sorry! I don't understand. Try asking about chatbots, AI or NLP."
        else:
            robo_response = sent_tokens[idx]

    except Exception as e:
        robo_response = "I am sorry! Something went wrong. Please try again."

    sent_tokens.remove(user_response)
    return robo_response

In [ ]:
flag = True

print("=" * 62)
print("🤖 ROBO: Hi! My name is Robo.")
print("🤖 ROBO: I will answer your queries about Chatbots, AI & NLP.")
print("🤖 ROBO: Type 'bye' to exit  |  'thanks' to end chat.")
print("=" * 62)
print()

while flag:
    try:
        user_response = input("You: ")
    except EOFError:
        break
    user_response = user_response.lower().strip()
    if not user_response:
        continue
    if user_response == 'bye':
        flag = False
        print("ROBO: Bye! Take care.. ")
    elif user_response in ('thanks', 'thank you'):
        flag = False
        print("ROBO: You are welcome! ")
    elif greeting(user_response) is not None:
        print("ROBO:", greeting(user_response))
    else:
        print("ROBO:", response(user_response))